# Giga Meter — Clean (data preparation)

Produces the **analysis-ready dataset** consumed by `gigameter_edaexplorer.ipynb`
and `gigameter_runbaseline.ipynb`. Run this first; the downstream notebooks load its
output instead of repeating the pull and the cleaning decisions.

**What it does:** pulls master / measurements / registration, prepares fields,
canonicalises ISP names, keeps the dominant M-Lab server, drops invalid rows,
applies the physical-validity cleaning, then two *data-informed* choices you make
here — the latency outlier cutoff and the school-hours window — followed by
preprocessing and a filter funnel showing exactly what was removed.

**What it writes** (to `CACHE_DIR`):
| File | Contents |
|---|---|
| `<slug>_clean.parquet` | analysis-ready measurements (`m`) |
| `<slug>_clean_unfiltered.parquet` | pre-outlier-filter snapshot (`m_original`) |
| `<slug>_clean_params.json` | every parameter + the filter funnel, so downstream notebooks inherit the decisions |

Master and registration keep their existing caches (`<ISO3>_master_datapull.csv`,
`<slug>_registered.parquet`) and are re-read downstream.

---
## Part 0 — Setup, load, clean

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pytz

from IPython.display import display

# Optional connectivity libs (only needed when USE_CACHED_DATA = False)
try:
    import delta_sharing
    DELTA_SHARING_AVAILABLE = True
except ImportError:
    DELTA_SHARING_AVAILABLE = False
    print("⚠️ delta_sharing not available - will use cached data only")

try:
    import trino
    from trino.dbapi import connect
    TRINO_AVAILABLE = True
except ImportError:
    TRINO_AVAILABLE = False
    print("⚠️ trino not available - will use cached data only")

# -----------------------------------------------------------------------------
# Data-loading helpers (bundled in ./helpers)
#   load_master            - school master via Delta Sharing / Trino, CSV-cached
#   load_measurements      - country measurements via Trino, parquet-cached + incremental
#   format_measurements    - query builder + light post-processing for the
#                            consolidated table default.all_gigameter_measurement_data
#   get_trino_cursor/engine - PRD Trino over kubectl port-forward (auto-started)
# -----------------------------------------------------------------------------
set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

try:
    import format_measurements
    from load_master import load_master, load_master_trino
    from load_measurements import (
        load_measurements,
        load_registration,
        get_trino_cursor,
        get_trino_engine,
    )
    HELPERS_AVAILABLE = True
except ImportError as e:
    HELPERS_AVAILABLE = False
    print(f"⚠️ data-loading helpers not importable from {set_up_dir}: {e}")

# -----------------------------------------------------------------------------
# Analysis helpers + Giga chart style (moved out of the notebook)
#   eda_helpers      - education inference, ISP canonicalisation, IQB-Edu engine,
#                      legacy service-tier scaffolding
#   giga_chart_style - fonts + palette + rcParams (applied on import)
# -----------------------------------------------------------------------------
from eda_helpers import (resolve_country, infer_edlevel_from_name, clean_isp, build_isp_canon,
                         IQB_CONFIG, IQB_USE_CASES, IQB_PERCENTILES, IQB_BENCHMARK,
                         MIN_MEASUREMENTS_FOR_IQB, calculate_iqb_score,
                         _config_for_use_case,
                         classify_service_level, tier_order,
                         TIER_THRESHOLD_1, TIER_THRESHOLD_2, TIER_THRESHOLD_3)
from giga_chart_style import (GIGA_PRIMARY, GIGA_GREY, GIGA_BLUE, GIGA_GOOD,
                              GIGA_MODERATE, GIGA_BAD, GIGA_TIER_RAMP, GIGA_CYCLE,
                              GIGA_SUPTITLE)

print("\u2713 Imports complete \u00b7 Giga chart style applied (Open Sans / Manrope, Giga palette)")

In [ ]:
# =============================================================================
# COUNTRY — set ONE code; iso2 / name / timezone resolve automatically
# =============================================================================
COUNTRY = "ZAF"   # ISO3 code or country name (registry: helpers/country_reference.json)

_c = resolve_country(COUNTRY)   # single-timezone countries pick their zone via pytz;
                                # multi-zone countries use the curated registry default.
COUNTRY_ISO3, COUNTRY_ISO2, COUNTRY_NAME, TIMEZONE = _c["iso3"], _c["iso2"], _c["name"], _c["timezone"]
# Override for multi-zone countries if needed: resolve_country(COUNTRY, timezone="Asia/Samarkand")

# ── Data loading ─────────────────────────────────────────────────────────────
USE_CACHED_DATA = True        # True: load parquet caches; False: pull/refresh from Trino
MEASUREMENT_SOURCE = None     # rt_source filter; None = all sources (safe default)

# Scale knobs (large countries, e.g. UZB ~4.7M rows). The parquet on disk ALWAYS
# keeps full history and all columns; these only scope what gets LOADED.
ROWLEVEL_WINDOW_DAYS = None   # e.g. 365 -> only load the trailing year of row-level data
LOAD_COLUMNS = None           # e.g. a column list -> prune columns at read

CACHE_DIR = f"./cache/{COUNTRY_NAME}"   # cleaned data + caches land here (gitignored)
# Master data caches as {ISO3}_master_datapull.csv in this directory

print(f"\u2713 {COUNTRY_NAME} ({COUNTRY_ISO3}/{COUNTRY_ISO2}) \u00b7 timezone {TIMEZONE}"
      + (f"  [country spans {len(_c['timezones'])} zones]" if len(_c['timezones']) > 1 else ""))

In [ ]:
# =============================================================================
# NOTEBOOK-LEVEL FILTERS & ANALYSIS PARAMETERS (independent of country loading)
# =============================================================================
ADMIN1_FILTER = None              # e.g. "Eastern Cape" scopes the whole notebook to one region
SCHOOL_HOURS_START = 8            # school-hours window (local time, 24h)
SCHOOL_HOURS_END = 16
MIN_WEEKDAYS_MEASURED = 10        # min weekdays with data for detailed per-school analysis
USE_EDUCATION_INFERENCE = False   # infer education_level from school names if govt field incomplete

# NOTE: the latency outlier threshold is NOT set here — it is picked at the
# preprocessing stage, informed by this country's latency distribution.

# Output options
EXPORT_RESULTS = False            # True -> export summary tables to CSV
OUTPUT_DIR = "./output"

print(f"  Admin1 filter: {ADMIN1_FILTER if ADMIN1_FILTER else 'None (all regions)'} · "
      f"school hours {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END} · "
      f"min weekdays {MIN_WEEKDAYS_MEASURED}")

In [ ]:
# =============================================================================
# ANALYSIS SCOPE — education level(s) and year(s) for the baseline analyses
# =============================================================================
# Every Q1-Q3 cell below is parameterised on these; nothing is hardcoded to
# 'Secondary' or a specific year. Set EDUCATION_LEVEL = None to analyse all
# schools. YEARS is auto-detected from the data unless you pin it.
EDUCATION_LEVEL = 'Secondary'   # e.g. 'Secondary', 'Primary', or None for all levels
YEARS = None                    # None -> auto-detect from the data; or e.g. ['2025', '2026']

# Thresholds used across the Ministry analyses
THR = 20        # download (Mbps) — the agreed contract threshold
THR_UL = 10     # upload (Mbps)
THR_LAT = 100   # latency (ms)
MIN_DAYS_MONTH = 10   # a school-month is "analysable" with >= this many measured days

print(f"Scope: education level = {EDUCATION_LEVEL or 'ALL'} · "
      f"years = {YEARS or 'auto-detect'} · thresholds {THR}/{THR_UL} Mbps, {THR_LAT} ms")

In [ ]:
# =============================================================================
# DATABASE CONNECTION (PRD Trino via kubectl port-forward)
# =============================================================================
# When USE_CACHED_DATA = False we query production Trino. The helpers below
# auto-start the tunnel if port 8080 isn't already open; you can also run it
# manually in a separate terminal and leave it open:
#
#     kubectl port-forward svc/trino 8080:8080 -n ictd-ooi-trino-prd
#
# Prereqs (one-time): az login -> az aks get-credentials --name uni-ooi-giga-aks-prd
#   --resource-group RS-UNI-GIGA-AKS-PRD -> kubelogin convert-kubeconfig -l azurecli
# If the tunnel fails to start, your az token has probably expired - re-run `az login`.
# =============================================================================
USE_CACHED_DATA = True

cur = None        # Trino DB-API cursor  -> used by load_measurements / refresh
engine = None     # SQLAlchemy engine    -> used by pd.read_sql for ad-hoc queries

if not USE_CACHED_DATA:
    cur = get_trino_cursor()      # auto-starts port-forward
    engine = get_trino_engine()   # auto-starts port-forward
    if cur is not None:
        print("✓ Trino PRD connection ready (catalog=delta_lake, schema=default)")
    else:
        print("⚠️ No Trino cursor - set USE_CACHED_DATA=True or fix the port-forward")
else:
    print("✓ Using cached data mode (no live Trino connection)")


In [ ]:
# =============================================================================
# LOAD MASTER DATA (load_master helper - Delta Sharing, CSV-cached)
# =============================================================================
# use_cached=True  -> read {ISO3}_master_datapull.csv from CACHE_DIR
# use_cached=False -> pull fresh from Delta Sharing and overwrite the cache
#                     (no port-forward needed - Delta Sharing is independent of Trino)

master_cache_csv = Path(CACHE_DIR) / f"{COUNTRY_ISO3}_master_datapull.csv"

master = load_master(COUNTRY_ISO3, master_cache_csv, use_cached=USE_CACHED_DATA)
master.head(3)


In [ ]:
# =============================================================================
# EDUCATION LEVEL NORMALIZATION
# =============================================================================

# Define inference function
def infer_edlevel_from_name(school_name):
    """Infer education level from school name patterns."""
    if pd.isna(school_name):
        return 'Unknown'
    
    name = str(school_name).upper()
    
    # Check for post-secondary
    if any(x in name for x in ['UNIVERSITY', 'COLLEGE', 'POLYTECHNIC', 'INSTITUTE']):
        return 'Post-secondary'
    
    # Check for secondary
    if any(x in name for x in ['SECONDARY', 'HIGH', 'TECHNICAL', 'VOCATIONAL']):
        return 'Secondary'
    
    # Check for primary and secondary combined
    if any(x in name for x in ['PRIMARY AND SECONDARY', 'PRIM. & SEC.', 'PRI. & SEC.']):
        return 'Primary and Secondary'
    
    # Check for primary
    if any(x in name for x in ['PRIMARY', 'BASIC', 'PRIM.', 'PRI.']):
        return 'Primary'
    
    # Check for pre-primary
    if any(x in name for x in ['PRE-PRIMARY', 'PRE-SCHOOL', 'ECE', 'NURSERY']):
        return 'Pre-primary'
    
    return 'Unknown'

print(f"\n{'='*80}")
print("EDUCATION LEVEL NORMALIZATION")
print(f"{'='*80}")

# Apply education level handling based on configuration
if USE_EDUCATION_INFERENCE:
    print(f"\nMode: Using inference from school names (education_level_inferred)")
    # Infer education level from school names
    master['education_level_inferred'] = master['school_name'].apply(infer_edlevel_from_name)
    
    # Prefer govt data where available, fall back to inferred
    master['education_level_normalized'] = master['education_level_govt'].fillna(master['education_level_inferred'])
    
    # Fill remaining nulls with 'Unknown'
    master['education_level_normalized'] = master['education_level_normalized'].fillna('Unknown')
    
    inferred_count = (master['education_level_normalized'] == master['education_level_inferred']).sum()
    govt_count = master['education_level_govt'].notna().sum()
    print(f"  Schools from govt data: {govt_count}")
    print(f"  Schools inferred from names: {inferred_count}")
    print(f"  Total schools: {len(master)}")
else:
    print(f"\nMode: Using government education level data only")
    # Just use govt data, fill nulls with Unknown
    master['education_level_normalized'] = master['education_level_govt'].fillna('Unknown')
    
    govt_count = master['education_level_govt'].notna().sum()
    unknown_count = (master['education_level_normalized'] == 'Unknown').sum()
    print(f"  Schools with govt data: {govt_count}")
    print(f"  Schools with unknown: {unknown_count}")
    print(f"  Total schools: {len(master)}")

# Display distribution
print(f"\nEducation Level Distribution:")
print(master['education_level_normalized'].value_counts().sort_index().to_string())


In [ ]:
# =============================================================================
# LOAD MEASUREMENTS (load_measurements helper - Trino, parquet-cached)
# =============================================================================
# Pulls from the consolidated table default.all_gigameter_measurement_data.
#   USE_CACHED_DATA = True   -> read the parquet cache (set in the CONFIG cell)
#   USE_CACHED_DATA = False  -> incremental refresh from Trino (delta since max cached date)
#   FORCE_REFRESH   = True   -> delete the parquet and pull EVERYTHING fresh from Trino
FORCE_REFRESH = False   # flip to True to wipe the cache and re-pull from scratch

if FORCE_REFRESH is True: 
    cur = None        # Trino DB-API cursor  -> used by load_measurements / refresh
    engine = None     # SQLAlchemy engine    -> used by pd.read_sql for ad-hoc queries

    cur = get_trino_cursor()      # auto-starts port-forward
    engine = get_trino_engine()   # auto-starts port-forward
    if cur is not None:
        print("✓ Trino PRD connection ready (catalog=delta_lake, schema=default)")
    else:
        print("⚠️ No Trino cursor - set USE_CACHED_DATA=True or fix the port-forward")
    
measurements_cache = Path(CACHE_DIR) / f"{COUNTRY_NAME.lower().replace(' ', '')}_measurements.parquet"

if FORCE_REFRESH and measurements_cache.exists():
    measurements_cache.unlink()
    print(f"\u2717 Deleted cache for full refresh: {measurements_cache.name}")

m = load_measurements(
    COUNTRY_NAME,
    measurements_cache,
    cur,
    use_cached=(USE_CACHED_DATA and not FORCE_REFRESH),   # FORCE_REFRESH always re-pulls
    source=MEASUREMENT_SOURCE,   # see CONFIG cell; None = all sources
    columns=globals().get('LOAD_COLUMNS'),            # scale knobs (CONFIG cell)
    window_days=globals().get('ROWLEVEL_WINDOW_DAYS'),
)
if globals().get('ROWLEVEL_WINDOW_DAYS'):
    print(f"NOTE: row-level data windowed to last {ROWLEVEL_WINDOW_DAYS} days - "
          f"full-history stats in this notebook reflect that window only.")

# Schema compatibility - the consolidated table's column set varies by source.
# GigaMeter rows use created_timestamp / isp_name / packet_loss_rate (no raw JSON);
# legacy DailyCheckApp rows also carry timestamp / detected_isp / results JSON.
# Alias the consolidated names to the legacy names the rest of the notebook expects.
if 'timestamp' not in m.columns and 'created_timestamp' in m.columns:
    m['timestamp'] = m['created_timestamp']
if 'detected_isp' not in m.columns and 'isp_name' in m.columns:
    m['detected_isp'] = m['isp_name']
if 'detected_isp_asn' not in m.columns and 'isp_asn' in m.columns:
    m['detected_isp_asn'] = m['isp_asn']

# Local-timezone conversion (downstream cells expect a tz-aware `timestamplocal`).
m['date'] = pd.to_datetime(m['date'])
m['timestamp'] = pd.to_datetime(m['timestamp'], utc=True)
m['timestamplocal'] = m['timestamp'].dt.tz_convert(TIMEZONE)

FILTER_LOG = {"loaded": len(m)}   # row-drop funnel, summarised at end of preprocessing

print(f"  Source(s): {m['rt_source'].value_counts().to_dict()}")
print(f"  Date range: {m['date'].min().date()} to {m['date'].max().date()}")
print(f"  Unique schools: {m['school_id_giga'].nunique()}")


In [ ]:
# =============================================================================
# LOAD REGISTRATION DATA - direct SQL + parquet (notebook level)
# =============================================================================
# One row per school from delta_lake.default.all_gigameter_registered_schools
# (already joined with admin/geo metadata + funnel: registered_gigameter,
# sending_gigameter_data, install_status, first/last_measurement_date, device counts).

registration_cache = Path(CACHE_DIR) / f"{COUNTRY_NAME.lower().replace(' ', '')}_registered.parquet"

registered_query = f"""
SELECT *
FROM default.all_gigameter_registered_schools
WHERE iso3_code = '{COUNTRY_ISO3.upper()}'
"""

if USE_CACHED_DATA and registration_cache.exists():
    r = pd.read_parquet(registration_cache)
    print(f"\u2713 Registration loaded from cache: {r.shape[0]:,} rows  ({registration_cache.name})")
else:
    # run the SQL directly
    cur.execute(registered_query)
    r = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    print(f"\u2713 Registration queried: {r.shape[0]:,} rows")
    # save to parquet
    registration_cache.parent.mkdir(parents=True, exist_ok=True)
    r.to_parquet(registration_cache, index=False)
    print(f"\u2713 Cached to {registration_cache.name}")

# r already carries admin1/admin2 - no master merge needed.
if ADMIN1_FILTER:
    print("\n  Schools by admin1 (before filter):")
    print(r['admin1'].value_counts().head())
    r = r[r['admin1'] == ADMIN1_FILTER]
    print(f"\n\u2713 Filtered to {ADMIN1_FILTER}: {r.shape[0]} schools")

print(f"  registry lists {len(r):,} schools  "
      f"(status labels install_status/sending_gigameter_data NOT used — "
      f"canonical 'sent data'/'live' is derived from measurements)")


In [ ]:
# =============================================================================
# MEASUREMENT FIELD PREP (physical table is already unpacked - no JSON parsing)
# =============================================================================
# all_gigameter_measurement_data delivers WiFi, server, ISP and loss fields
# pre-extracted. We only coerce numeric dtypes and expose loss_rate.

# WiFi metrics - some arrive as object/string; coerce to numeric.
for c in ['detected_wifi_quality', 'detected_wifi_signal', 'detected_wifi_tx_rate',
          'detected_wifi_channel', 'detected_wifi_frequency']:
    if c in m.columns:
        m[c] = pd.to_numeric(m[c], errors='coerce')

m['loss_rate'] = pd.to_numeric(m['packet_loss_rate'], errors='coerce')

_wifi_n = m['detected_wifi_ssid'].notna().sum() if 'detected_wifi_ssid' in m.columns else 0
print(f"✓ WiFi info: {_wifi_n:,} measurements")
if 'detected_server' in m.columns:
    print(f"✓ Server (detected_server): {m['detected_server'].notna().sum():,} measurements")
print(f"✓ Packet loss: {m['loss_rate'].notna().sum():,} measurements "
      f"({100 * m['loss_rate'].notna().mean():.0f}% coverage, median {m['loss_rate'].median():.4%})")

In [ ]:
# =============================================================================
# ISP NAME NORMALISATION  ->  isp_mapped: keep only name, join Telkom, MTN, Vodacom variants
# =============================================================================

import re

def simplify_isp_name(isp):
    """Simplify ISP name to just the core name, combining common variants."""
    if not isinstance(isp, str):
        return None
    isp_l = isp.lower()
    if re.search(r"mtn", isp_l):
        return "MTN"
    if re.search(r"telkom", isp_l):
        return "Telkom"
    if re.search(r"vodacom", isp_l):
        return "Vodacom"
    if re.search(r"cell ?c", isp_l):
        return "Cell C"
    # Add more consolidation as needed
    # fallback: remove ASN/AS numbers, legal pieces, keep main name part
    # e.g., "AS328471 HERO TELECOMS (PTY)" -> "HERO TELECOMS"
    # remove ASN codes
    isp_clean = re.sub(r"AS\d{4,}\s*", "", isp)
    # remove legal entity suffixes
    isp_clean = re.sub(r"\s*\(pty\)|\s*soc|\s*ltd\.?|pty\.?\s*limited|pty|limited|soc", "", isp_clean, flags=re.IGNORECASE)
    # remove extra spaces
    isp_clean = isp_clean.strip()
    return isp_clean if isp_clean else isp

_isp_src = "detected_isp" if "detected_isp" in m.columns else "isp_name"
m["isp_mapped"] = m[_isp_src].apply(simplify_isp_name)
print(f"✓ isp_mapped: {m['isp_mapped'].nunique()} grouped ISPs (from {m[_isp_src].nunique()} raw variants)")
print(m["isp_mapped"].value_counts().head(10).to_string())

In [ ]:
# =============================================================================
# SERVER SELECTION — keep measurements against the MAIN test server(s), defined by % of measurements
# =============================================================================
# M-Lab routing can send tests to different servers over time; mixing servers
# mixes baseline latency (server distance), so downstream latency comparisons
# use the dominant servers only. Rows with no server info are kept.
# NOTE: detected_server comes from a stale mlab-ns endpoint and is imperfect —
# treat it as a routing hint, not ground truth. Set SERVER_FILTER = False to keep all.
# SERVER_PERC_THRESHOLD defines the percent cutoff (e.g., 0.80 for servers with >= 80% of traffic)
SERVER_FILTER = True
SERVER_PERC_THRESHOLD = 0.30  # Adjust this threshold as needed

_srv = m['detected_server'].value_counts(dropna=False)
total_with_server = _srv.drop(index=[np.nan]) if np.nan in _srv.index else _srv
total_count = total_with_server.sum()
print("Measurements per detected server (median latency):")
for _s, _n in _srv.head(8).items():
    _lat = m.loc[m['detected_server'] == _s, 'latency'].median()
    print(f"  {str(_s)[:28]:28} {_n:>8,}   {_lat:5.0f} ms")
_no_srv = m['detected_server'].isna().sum()
print(f"  {'(no server info)':28} {_no_srv:>8,}")

# Determine dominant server(s) by cumulative percentage
_srv_frac = (_srv / _srv.sum())
# Only want to consider actual servers, not NaN
_srv_nonan = _srv[~_srv.index.isna()]
_srv_frac_nonan = (_srv_nonan / _srv_nonan.sum())

# servers meeting the min percentage threshold
main_servers = _srv_frac_nonan[_srv_frac_nonan >= SERVER_PERC_THRESHOLD].index.tolist()

if SERVER_FILTER and main_servers:
    _b = len(m)
    m = m[(m['detected_server'].isin(main_servers)) | m['detected_server'].isna()]
    FILTER_LOG['other_servers'] = _b - len(m)
    print(f"\n✓ MAIN_SERVERS (>{SERVER_PERC_THRESHOLD*100:.0f}% threshold): {main_servers}")
    print(f"  Kept {len(m):,} rows ({_b - len(m):,} removed; no-server rows kept)")
else:
    FILTER_LOG['other_servers'] = 0
    print("\n(server filter off — all servers kept)")

In [ ]:
# =============================================================================
# MERGE MASTER METADATA INTO MEASUREMENTS
# =============================================================================
# The consolidated table already carries admin1/admin2/connectivity_type_govt/
# latitude/longitude/education_level. Only merge the master-derived columns that
# aren't already on `m`, to avoid _x/_y suffix collisions on re-merge. This works
# whether `m` came from the new 80+ col table or an older cached parquet.

_wanted = ['education_level', 'education_level_govt', 'connectivity_provider',
           'admin1', 'admin2', 'connectivity_type_govt', 'latitude', 'longitude']
_merge_cols = ['school_id_giga'] + [c for c in _wanted if c in master.columns and c not in m.columns]
m = m.merge(master[_merge_cols], on='school_id_giga', how='left')
print(f"✓ Merged master columns into m: {[c for c in _merge_cols if c != 'school_id_giga']}")


In [ ]:
# =============================================================================
# FILTER INVALID MEASUREMENTS (future-dated & other quality issues)
# =============================================================================
_n_before_invalid = len(m)

# Remove future-dated measurements (data quality issue - year 2247 etc)
tomorrow = (pd.Timestamp.now(tz='UTC') + pd.Timedelta(days=1)).normalize()
future_count = (m['timestamplocal'].dt.tz_convert('UTC') >= tomorrow).sum()

if future_count > 0:
    print(f"⚠️   Removing {future_count:,} future-dated measurements (data corruption):")
    future_examples = m[m['timestamplocal'].dt.tz_convert('UTC') >= tomorrow]['timestamplocal'].head(3)
    for ts in future_examples:
        print(f"      {ts}")
    m = m[m['timestamplocal'].dt.tz_convert('UTC') < tomorrow]
    print(f"✓ Kept {len(m):,} valid measurements")
else:
    print(f"✓ No future-dated measurements found")

# -----------------------------------------------------------------------------
# PHYSICAL-VALIDITY CLEANING (added 2026-07-30)
# Negative speeds/latency are impossible; latency around 4,294,967 ms
# (= 2^32 microseconds) is a uint32-overflow sentinel, not a measurement.
for _c in ['download_speed', 'upload_speed', 'latency']:
    if _c in m.columns:
        _v = pd.to_numeric(m[_c], errors='coerce')
        _bad = _v < 0
        if _c == 'latency':
            _bad |= _v >= 4_294_967
        if int(_bad.sum()):
            print(f"\u2713 Cleaned {_c}: {int(_bad.sum()):,} invalid values -> NaN")
        m[_c] = _v.mask(_bad)

FILTER_LOG['future_dated'] = _n_before_invalid - len(m)
# (physical-validity cleaning nulls invalid VALUES; it does not drop rows)


In [ ]:
# =============================================================================
# PRESERVE ORIGINAL MEASUREMENT DATA (clean, unfiltered)
# =============================================================================

# Create copy for drop-off/time-based analysis (after removing invalid timestamps, before time/admin filtering)
m_original = m.copy()

In [ ]:
# =============================================================================
# LATENCY CUTOFF ANALYSIS - COMPARING METHODS
# =============================================================================

print("\n" + "="*80)
print("METHOD 1: INTERQUARTILE RANGE (IQR)")
print("="*80)

latency_q1 = m['latency'].quantile(0.25)
latency_q3 = m['latency'].quantile(0.75)
latency_iqr = latency_q3 - latency_q1
latency_iqr_threshold = latency_q3 + 1.5 * latency_iqr

iqr_outliers = (m['latency'] > latency_iqr_threshold).sum()
iqr_pct = iqr_outliers / len(m) * 100

print(f"Q1 (25th percentile): {latency_q1:>8.0f}ms")
print(f"Q3 (75th percentile): {latency_q3:>8.0f}ms")
print(f"IQR (Q3 - Q1):        {latency_iqr:>8.0f}ms")
print(f"\nThreshold (Q3 + 1.5×IQR): {latency_iqr_threshold:>8.0f}ms")
print(f"Outliers flagged: {iqr_outliers:,} ({iqr_pct:.2f}% of data)")

print("\n" + "="*80)
print("METHOD 2: MODIFIED Z-SCORE (MAD - Median Absolute Deviation)")
print("="*80)

latency_median = m['latency'].median()
latency_mad = (m['latency'] - latency_median).abs().median()   # NaN-skipping (runs pre-filter)
latency_modified_z_threshold = latency_median + 3.5 * latency_mad / 0.6745

mz_outliers = (m['latency'] > latency_modified_z_threshold).sum()
mz_pct = mz_outliers / len(m) * 100

print(f"Median latency:       {latency_median:>8.0f}ms")
print(f"MAD:                  {latency_mad:>8.0f}ms")
print(f"\nThreshold (median + 3.5×MAD/0.6745): {latency_modified_z_threshold:>8.0f}ms")
print(f"Outliers flagged: {mz_outliers:,} ({mz_pct:.2f}% of data)")

print("\n" + "="*80)
print("METHOD 3: PERCENTILE-BASED")
print("="*80)

p95_threshold = m['latency'].quantile(0.95)
p99_threshold = m['latency'].quantile(0.99)
p999_threshold = m['latency'].quantile(0.999)

p95_outliers = (m['latency'] > p95_threshold).sum()
p99_outliers = (m['latency'] > p99_threshold).sum()
p999_outliers = (m['latency'] > p999_threshold).sum()

print(f"95th percentile: {p95_threshold:>8.0f}ms → {p95_outliers:,} outliers ({p95_outliers/len(m)*100:.2f}%)")
print(f"99th percentile: {p99_threshold:>8.0f}ms → {p99_outliers:,} outliers ({p99_outliers/len(m)*100:.2f}%)")
print(f"99.9th percentile: {p999_threshold:>8.0f}ms → {p999_outliers:,} outliers ({p999_outliers/len(m)*100:.2f}%)")

print("\n" + "="*80)
print("METHOD 4: BY CONNECTIVITY TYPE (Modified Z-score per type)")
print("="*80)

type_thresholds = {}
for conn_type in m['connectivity_type_govt'].dropna().unique():
    subset = m[m['connectivity_type_govt'] == conn_type]['latency'].dropna()
    if len(subset) > 0:
        med = subset.median()
        mad = np.median(np.abs(subset - med))
        threshold = med + 3.5 * mad / 0.6745
        outliers = (subset > threshold).sum()
        pct = outliers / len(subset) * 100
        type_thresholds[conn_type] = threshold

        print(f"\n{conn_type} (n={len(subset):,}):")
        print(f"  Median: {med:>8.0f}ms")
        print(f"  Threshold: {threshold:>8.0f}ms")
        print(f"  Outliers: {outliers:,} ({pct:.2f}%)")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

summary_data = {
    'IQR (1.5×)': (latency_iqr_threshold, iqr_pct),
    'Modified Z-score (3.5×MAD)': (latency_modified_z_threshold, mz_pct),
    '95th percentile': (p95_threshold, p95_outliers/len(m)*100),
    '99th percentile': (p99_threshold, p99_outliers/len(m)*100),
}

print("\nComparison of methods (all data):")
for method, (threshold, pct) in summary_data.items():
    print(f"  {method:40} → {threshold:>8.0f}ms (filters {pct:>6.2f}%)")

# Visualize all thresholds
fig, ax = plt.subplots(figsize=(14, 6))

# Histogram of valid latency data (0-1000ms)
valid_latency = m[(m['latency'] > 0) & (m['latency'] < 1000)]['latency']
ax.hist(valid_latency.dropna(), bins=100, alpha=0.6, color='#0050e6', edgecolor='black', label='Valid latency data')

# Add threshold lines
thresholds_to_plot = [
    (latency_iqr_threshold, 'IQR (1.5×)', '#989898'),
    (latency_modified_z_threshold, 'Modified Z', '#002d9c'),
    (p99_threshold, '99th percentile', '#7eb0ff'),
]

for threshold, label, color in thresholds_to_plot:
    if threshold < 1000:  # Only plot if in visible range
        ax.axvline(threshold, color=color, linestyle='--', linewidth=2, label=f'{label}: {threshold:.0f}ms')

ax.set_xlabel('Latency (ms)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Latency Distribution with Candidate Outlier Thresholds', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1000])
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# LATENCY OUTLIER THRESHOLD — inspect the distribution, then pick
# =============================================================================
# The preprocessing cell below applies this threshold globally. Default = p99
# for THIS country's distribution; review the histogram and candidates, then
# override the assignment at the bottom if a different cutoff fits better.
_lat = m['latency'].dropna()
_candidates = {'p95': _lat.quantile(0.95), 'p99': _lat.quantile(0.99),
               'p99.5': _lat.quantile(0.995),
               'IQR (Q3+1.5×IQR)': latency_iqr_threshold,        # from the method comparison above
               'mod-z (3.5×MAD)': latency_modified_z_threshold,  # from the method comparison above
               'fixed 400': 400, 'fixed 1000': 1000}

fig, ax = plt.subplots(figsize=(11, 3.5))
_plot = _lat[_lat <= _lat.quantile(0.999)]
ax.hist(_plot, bins=120, color=GIGA_PRIMARY[600], alpha=0.85)
for (_name, _v), _c in zip(_candidates.items(),
                           [GIGA_PRIMARY[300], GIGA_GREY[700], GIGA_PRIMARY[800],
                            GIGA_GOOD, GIGA_CYCLE[3], GIGA_MODERATE, GIGA_BAD]):
    if _v <= _plot.max():
        ax.axvline(_v, color=_c, linestyle='--', linewidth=1.3, label=f'{_name}: {_v:.0f} ms')
ax.set_yscale('log')
ax.set_xlabel('Latency (ms; display clipped at p99.9)')
ax.set_ylabel('measurements (log)')
ax.set_title(f'Latency distribution — pick the outlier cutoff — {COUNTRY_NAME}')
ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

print('Candidate cutoffs and the share of measurements each would exclude:')
for _name, _v in _candidates.items():
    print(f"  {_name:>10}: {_v:7.0f} ms -> excludes {100 * (_lat > _v).mean():5.2f}%")

LATENCY_OUTLIER_THRESHOLD = round(float(_lat.quantile(0.99)))   # default: p99 — OVERRIDE after review
# LATENCY_OUTLIER_THRESHOLD = 

print(f"\nLATENCY_OUTLIER_THRESHOLD = {LATENCY_OUTLIER_THRESHOLD} ms (default = p99; edit this line to override)")

In [ ]:
# =============================================================================
# TIME-OF-DAY PROFILE — inspect, then set the school-hours window (per-country)
# =============================================================================
# The preprocessing cell below applies SCHOOL_HOURS_START/END (from the filters
# cell) to classify school_hours vs off_hours — the cut behind per-school IQB
# and every "school hours" analysis (~79% of measurements in Fiji). Review the
# histogram and override the window HERE if this country's school day differs.
_hr = m['timestamplocal'].dt.hour

SCHOOL_HOURS_START, SCHOOL_HOURS_END = 7, 16   # <- override, then run on

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.hist(_hr, bins=range(25), color=GIGA_PRIMARY[600], alpha=0.85, edgecolor='white')
ax.axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END, color=GIGA_GOOD, alpha=0.12)
ax.axvline(SCHOOL_HOURS_START, color=GIGA_GREY[700], ls='--', lw=1.2)
ax.axvline(SCHOOL_HOURS_END, color=GIGA_GREY[700], ls='--', lw=1.2)
ax.set_xticks(range(0, 25, 2)); ax.set_xlabel('local hour'); ax.set_ylabel('measurements')
ax.set_title(f'Measurements by local time of day — {COUNTRY_NAME} '
             f'(shaded = school hours {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END})')
plt.tight_layout(); plt.show()

_n_sch = _hr.between(SCHOOL_HOURS_START, SCHOOL_HOURS_END).sum()
print(f"Window {SCHOOL_HOURS_START}-{SCHOOL_HOURS_END} captures {_n_sch:,} of {len(m):,} "
      f"measurements ({100 * _n_sch / len(m):.0f}%) — applied by the preprocessing cell below.")

In [ ]:
# =============================================================================
# DATA PREPROCESSING
# =============================================================================
# Add measurement date and weekday columns
m['measurement_date'] = pd.to_datetime(m['timestamplocal']).dt.date
m['measurement_weekday'] = pd.to_datetime(m['timestamplocal']).dt.weekday  # 0=Monday

# Apply admin filter if specified
_n_before_admin = len(m)
if ADMIN1_FILTER:
    m = m[m.admin1 == ADMIN1_FILTER]
    print(f"✓ Measurements filtered to {ADMIN1_FILTER}: {m.shape[0]} records")
FILTER_LOG['admin1_filter'] = _n_before_admin - len(m)

# Filter latency outliers (threshold chosen above, from the distribution).
# Rows with MISSING latency are kept — they still carry valid speed tests.
_before = len(m)
m = m[(m['latency'] < LATENCY_OUTLIER_THRESHOLD) | m['latency'].isna()]
FILTER_LOG['latency_outliers'] = _before - len(m)
print(f"✓ Removed {_before - len(m):,} latency outliers (>={LATENCY_OUTLIER_THRESHOLD} ms; NaN latency kept)")

# Add time window classification
def classify_time_window(hour):
    if SCHOOL_HOURS_START <= hour <= SCHOOL_HOURS_END:
        return 'school_hours'
    return 'off_hours'

m['measurement_time_window'] = m['timestamplocal'].dt.hour.apply(classify_time_window)

In [ ]:
# =============================================================================
# PREPROCESSING FUNNEL — what was filtered out of the analysis
# =============================================================================
_final = len(m)
_stages = [(k, v) for k, v in FILTER_LOG.items() if k != 'loaded']
print("=" * 64)
print("MEASUREMENTS FILTERED OUT OF THE ANALYSIS")
print("=" * 64)
print(f"  {'Loaded from cache/Trino':32s} {FILTER_LOG['loaded']:>10,}")
for _k, _v in _stages:
    print(f"  − {_k.replace('_', ' '):30s} {_v:>10,}   ({100 * _v / FILTER_LOG['loaded']:.2f}%)")
_dropped = FILTER_LOG['loaded'] - _final
print("-" * 64)
print(f"  {'ANALYSED':32s} {_final:>10,}   ({100 * _final / FILTER_LOG['loaded']:.1f}% of loaded; "
      f"{_dropped:,} rows removed)")
if 'measurement_time_window' in m.columns:
    _off = int((m['measurement_time_window'] != 'school_hours').sum())
    print(f"\n  Of the analysed rows, {_off:,} are OFF-HOURS ({100 * _off / _final:.0f}%) — kept in m,")
    print("  but excluded from every school-hours analysis (per-school IQB, performance")
    print(f"  distributions): those run on {_final - _off:,} school-hours measurements.")

print("\n  Note: physical-validity cleaning (negative speeds, latency overflow")
print("  sentinel) nulls VALUES without dropping rows, so it is not listed here.")

---
## Export the clean dataset

In [ ]:
# =============================================================================
# EXPORT — analysis-ready dataset + parameters for downstream notebooks
# =============================================================================
import json as _json
from datetime import datetime as _dt

_slug = COUNTRY_NAME.lower().replace(' ', '')
_clean_path = Path(CACHE_DIR) / f"{_slug}_clean.parquet"
_unfilt_path = Path(CACHE_DIR) / f"{_slug}_clean_unfiltered.parquet"
_params_path = Path(CACHE_DIR) / f"{_slug}_clean_params.json"

m.to_parquet(_clean_path, index=False)
m_original.to_parquet(_unfilt_path, index=False)

_params = {
    'country': {'iso3': COUNTRY_ISO3, 'iso2': COUNTRY_ISO2, 'name': COUNTRY_NAME,
                'timezone': TIMEZONE},
    'filters': {'admin1_filter': ADMIN1_FILTER,
                'measurement_source': MEASUREMENT_SOURCE,
                'main_servers': ([str(_x) for _x in main_servers] if 'main_servers' in globals()
                     else ([str(MAIN_SERVER)] if 'MAIN_SERVER' in globals() and MAIN_SERVER else None)),
                'server_pct_threshold': globals().get('SERVER_PERC_THRESHOLD'),
                'server_filter': bool(globals().get('SERVER_FILTER', False)),
                'latency_outlier_threshold_ms': float(LATENCY_OUTLIER_THRESHOLD),
                'school_hours_start': SCHOOL_HOURS_START,
                'school_hours_end': SCHOOL_HOURS_END},
    'scope_defaults': {'education_level': EDUCATION_LEVEL, 'years': YEARS,
                       'min_days_month': MIN_DAYS_MONTH,
                       'thr_download_mbps': THR, 'thr_upload_mbps': THR_UL,
                       'thr_latency_ms': THR_LAT,
                       'min_weekdays_measured': MIN_WEEKDAYS_MEASURED},
    'filter_funnel': {k: int(v) for k, v in FILTER_LOG.items()},
    'rows': {'analysed': int(len(m)), 'unfiltered': int(len(m_original)),
             'schools': int(m['school_id_giga'].nunique())},
    'window': {'first_date': str(m['date'].min().date()), 'last_date': str(m['date'].max().date())},
    'generated_at': _dt.now().isoformat(timespec='seconds'),
}
_params_path.write_text(_json.dumps(_params, indent=2))

print(f"\u2713 {_clean_path.name}       {len(m):,} rows x {len(m.columns)} cols "
      f"({_clean_path.stat().st_size/1e6:.1f} MB)")
print(f"\u2713 {_unfilt_path.name}  {len(m_original):,} rows "
      f"({_unfilt_path.stat().st_size/1e6:.1f} MB)")
print(f"\u2713 {_params_path.name}")
print(_json.dumps(_params['filters'], indent=2))